In [2]:
#Step 1 — Import libraries
import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import callbacks

print("Libraries imported successfully.")
print("TensorFlow version:", tf.__version__)

Libraries imported successfully.
TensorFlow version: 2.20.0


In [3]:
# Step 2 — Load the dataset

DATA_PATH = "/kaggle/input/datasets/yasserh/loan-default-dataset/Loan_Default.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

Dataset loaded successfully.
Shape: (148670, 34)

First 5 rows:


,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,24891,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,...,EQUI,552,EXP,55-64,to_inst,NaN,North,direct,1,NaN
2,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,...,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


In [4]:
# Step 3 — Separate target and features

# Check target column and class distribution

print("Target column exists:", "Status" in df.columns)

print("\nTarget distribution:")
print(df["Status"].value_counts())

print("\nTarget distribution (%):")
print((df["Status"].value_counts(normalize=True) * 100).round(2))

print("\nAll columns:")
print(df.columns.tolist())

Target column exists: True

Target distribution:
Status
0    112031
1     36639
Name: count, dtype: int64

Target distribution (%):
Status
0    75.36
1    24.64
Name: proportion, dtype: float64

All columns:
['ID', 'year', 'loan_limit', 'Gender', 'approv_in_adv', 'loan_type', 'loan_purpose', 'Credit_Worthiness', 'open_credit', 'business_or_commercial', 'loan_amount', 'rate_of_interest', 'Interest_rate_spread', 'Upfront_charges', 'term', 'Neg_ammortization', 'interest_only', 'lump_sum_payment', 'property_value', 'construction_type', 'occupancy_type', 'Secured_by', 'total_units', 'income', 'credit_type', 'Credit_Score', 'co-applicant_credit_type', 'age', 'submission_of_application', 'LTV', 'Region', 'Security_Type', 'Status', 'dtir1']


In [5]:
# Step 4 — Remove year and separate features from target

# Remove the 'year' column
df = df.drop("year", axis=1)

# Separate features and target
X = df.drop("Status", axis=1)
y = df["Status"]

print("After removing 'year':")
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nNumber of input features:", X.shape[1])

After removing 'year':
X shape: (148670, 32)
y shape: (148670,)

Target distribution:
Status
0    112031
1     36639
Name: count, dtype: int64

Number of input features: 32


In [6]:
print(y.value_counts(normalize=True))

Status
0    0.753555
1    0.246445
Name: proportion, dtype: float64


In [7]:
# Step 5 — Identify numerical and categorical features

# Identify numerical and categorical features

numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Number of numerical features:", len(numerical_features))
print("Numerical features:")
print(numerical_features)

print("\nNumber of categorical features:", len(categorical_features))
print("Categorical features:")
print(categorical_features)

Number of numerical features: 11
Numerical features:
['ID', 'loan_amount', 'rate_of_interest', 'Interest_rate_spread', 'Upfront_charges', 'term', 'property_value', 'income', 'Credit_Score', 'LTV', 'dtir1']

Number of categorical features: 21
Categorical features:
['loan_limit', 'Gender', 'approv_in_adv', 'loan_type', 'loan_purpose', 'Credit_Worthiness', 'open_credit', 'business_or_commercial', 'Neg_ammortization', 'interest_only', 'lump_sum_payment', 'construction_type', 'occupancy_type', 'Secured_by', 'total_units', 'credit_type', 'co-applicant_credit_type', 'age', 'submission_of_application', 'Region', 'Security_Type']


In [8]:
# Step 6 — Split into training and testing data

# Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting set:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training set:
X_train: (118936, 32)
y_train: (118936,)

Testing set:
X_test: (29734, 32)
y_test: (29734,)

Training target distribution:
Status
0    89625
1    29311
Name: count, dtype: int64

Testing target distribution:
Status
0    22406
1     7328
Name: count, dtype: int64


In [9]:
# Step 7 — Create the preprocessing pipeline

# Numerical preprocessing:
# 1. Fill missing numerical values with the median
# 2. Standardize numerical values

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# Categorical preprocessing:
# 1. Fill missing categorical values with the most frequent value
# 2. Convert categories into one-hot encoded numerical values

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])


# Combine both preprocessing pipelines

preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [10]:
# Step 8 — Fit the preprocessor and transform the data
# Fit preprocessing only on the training data
preprocessor.fit(X_train)

# Transform training and testing data
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Preprocessing completed successfully.")

print("\nOriginal shapes:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nProcessed shapes:")
print("X_train_processed:", X_train_processed.shape)
print("X_test_processed :", X_test_processed.shape)

print("\nProcessed data type:", type(X_train_processed))

Preprocessing completed successfully.

Original shapes:
X_train: (118936, 32)
X_test : (29734, 32)

Processed shapes:
X_train_processed: (118936, 70)
X_test_processed : (29734, 70)

Processed data type: <class 'numpy.ndarray'>


In [11]:
# Verify processed data

print("Missing values in X_train_processed:",
      np.isnan(X_train_processed).sum())

print("Missing values in X_test_processed:",
      np.isnan(X_test_processed).sum())

print("\nInfinite values in X_train_processed:",
      np.isinf(X_train_processed).sum())

print("Infinite values in X_test_processed:",
      np.isinf(X_test_processed).sum())

print("\nFinal input size for neural network:",
      X_train_processed.shape[1])

Missing values in X_train_processed: 0
Missing values in X_test_processed: 0

Infinite values in X_train_processed: 0
Infinite values in X_test_processed: 0

Final input size for neural network: 70


In [12]:
# Step 10 — Build the Deep Learning model

# Build the Deep Learning model

model = Sequential([
    Input(shape=(X_train_processed.shape[1],)),

    Dense(32, kernel_initializer="uniform", activation="relu"),
    Dense(32, kernel_initializer="uniform", activation="relu"),
    Dense(16, kernel_initializer="uniform", activation="relu"),

    Dropout(0.25),

    Dense(8, kernel_initializer="uniform", activation="relu"),

    Dropout(0.50),

    Dense(1, kernel_initializer="uniform", activation="sigmoid")
])

# Compile the model
optimizer = Adam(learning_rate=0.00009)

model.compile(
    optimizer=optimizer,
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Display model architecture
model.summary()

2026-09-21 13:32:06.457251: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │         2,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,001 (15.63 KB)

 Trainable params: 4,001 (15.63 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
# Step 11 — Configure Early Stopping

# Configure Early Stopping

early_stopping = callbacks.EarlyStopping(
    monitor="val_loss",
    min_delta=0.001,
    patience=10,
    restore_best_weights=True
)

print("Early stopping configured successfully.")
print("Monitor:", early_stopping.monitor)
print("Patience:", early_stopping.patience)
print("Restore best weights:", early_stopping.restore_best_weights)

Early stopping configured successfully.
Monitor: val_loss
Patience: 10
Restore best weights: True


In [15]:
# Find an actual test example whose true Status is 1
sample_index = y_test[y_test == 1].index[0]

sample = X_test.loc[sample_index]

print("Sample index:", sample_index)
print("\nActual Status:", y_test.loc[sample_index])
print("\nSample:")
print(sample)

Sample index: 4321

Actual Status: 1

Sample:
ID                               29211
loan_limit                         ncf
Gender                            Male
approv_in_adv                    nopre
loan_type                        type1
loan_purpose                        p1
Credit_Worthiness                   l1
open_credit                       nopc
business_or_commercial           nob/c
loan_amount                    3576500
rate_of_interest                   NaN
Interest_rate_spread               NaN
Upfront_charges                    NaN
term                             360.0
Neg_ammortization              not_neg
interest_only                  not_int
lump_sum_payment              not_lpsm
property_value               5208000.0
construction_type                   sb
occupancy_type                      pr
Secured_by                        home
total_units                         1U
income                        119340.0
credit_type                        EXP
Credit_Score      

In [16]:
sample_processed = preprocessor.transform(
    X_test.loc[[sample_index]]
)

if hasattr(sample_processed, "toarray"):
    sample_processed = sample_processed.toarray()

probability = model.predict(
    sample_processed,
    verbose=0
)[0][0]

print("Actual Status:", y_test.loc[sample_index])
print("Predicted probability:", probability)
print("Predicted class:", int(probability >= 0.5))

Actual Status: 1
Predicted probability: 0.70316076
Predicted class: 1


In [14]:
# Step 12 — Train the neural network

# Train the Deep Learning model

history = model.fit(
    X_train_processed,
    y_train,
    batch_size=32,
    epochs=50,
    validation_split=0.20,
    callbacks=[early_stopping],
    verbose=1
)

print("\nTraining completed.")
print("Epochs actually trained:", len(history.history["loss"]))

Epoch 1/50
2974/2974 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.7535 - loss: 0.5187 - val_accuracy: 0.7541 - val_loss: 0.4193
Epoch 2/50
2974/2974 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8602 - loss: 0.3680 - val_accuracy: 0.9224 - val_loss: 0.2575
Epoch 3/50
2974/2974 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9593 - loss: 0.2068 - val_accuracy: 0.9919 - val_loss: 0.1436
Epoch 4/50
2974/2974 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9901 - loss: 0.1358 - val_accuracy: 0.9982 - val_loss: 0.1101
Epoch 5/50
2974/2974 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9922 - loss: 0.1089 - val_accuracy: 0.9989 - val_loss: 0.0892
Epoch 6/50
 106/2974 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - accuracy: 0.9895 - loss: 0.1019

KeyboardInterrupt: 

In [15]:
# Step 13 — Evaluate on the untouched test set

# Evaluate the trained model on the untouched test set

test_loss, test_accuracy = model.evaluate(
    X_test_processed,
    y_test,
    verbose=0
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

Test Loss: 0.010028395801782608
Test Accuracy: 0.9988565444946289


In [16]:
# Step 14 — Generate predictions and confusion matrix

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    f1_score
)

# Get predicted probabilities
y_prob = model.predict(X_test_processed, verbose=0).ravel()

# Convert probabilities to binary predictions
y_pred = (y_prob >= 0.5).astype(int)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Confusion Matrix:
[[22375    31]
 [    3  7325]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9999    0.9986    0.9992     22406
           1     0.9958    0.9996    0.9977      7328

    accuracy                         0.9989     29734
   macro avg     0.9978    0.9991    0.9985     29734
weighted avg     0.9989    0.9989    0.9989     29734

F1 Score: 0.9976845546172705
ROC-AUC: 0.9997624750437247


In [17]:
# Step 15 — Check the actual target meaning

# Check the original Status values and their meaning in the dataset

print("Status values:")
print(sorted(df["Status"].unique()))

print("\nStatus 0 count:", (df["Status"] == 0).sum())
print("Status 1 count:", (df["Status"] == 1).sum())

print("\nStatus data type:", df["Status"].dtype)

Status values:
[np.int64(0), np.int64(1)]

Status 0 count: 112031
Status 1 count: 36639

Status data type: int64


In [18]:
# Step 16 — Create the model output directory

# Create directory for deployment artifacts

MODEL_DIR = "loan_default_model"

os.makedirs(MODEL_DIR, exist_ok=True)

print("Model directory created:", MODEL_DIR)
print("Directory exists:", os.path.exists(MODEL_DIR))

Model directory created: loan_default_model
Directory exists: True


In [19]:
# Step 17 — Save the trained neural network

# Save the trained neural network

MODEL_PATH = os.path.join(MODEL_DIR, "loan_default_nn.keras")

model.save(MODEL_PATH)

print("Model saved successfully.")
print("Model path:", MODEL_PATH)
print("File exists:", os.path.exists(MODEL_PATH))
print("File size:", round(os.path.getsize(MODEL_PATH) / (1024 * 1024), 2), "MB")

Model saved successfully.
Model path: loan_default_model/loan_default_nn.keras
File exists: True
File size: 0.08 MB


In [20]:
# Step 18 — Save the preprocessing pipeline with Joblib

# Save the preprocessing pipeline using Joblib

PREPROCESSOR_PATH = os.path.join(MODEL_DIR, "preprocessor.pkl")

joblib.dump(preprocessor, PREPROCESSOR_PATH)

print("Preprocessor saved successfully.")
print("Preprocessor path:", PREPROCESSOR_PATH)
print("File exists:", os.path.exists(PREPROCESSOR_PATH))
print("File size:", round(os.path.getsize(PREPROCESSOR_PATH) / (1024 * 1024), 2), "MB")

Preprocessor saved successfully.
Preprocessor path: loan_default_model/preprocessor.pkl
File exists: True
File size: 0.01 MB


In [21]:
# Step 19 — Load the saved model and preprocessor

# Load the saved model and preprocessor

from tensorflow.keras.models import load_model

loaded_model = load_model(MODEL_PATH)
loaded_preprocessor = joblib.load(PREPROCESSOR_PATH)

print("Saved model loaded successfully.")
print("Saved preprocessor loaded successfully.")

print("\nLoaded model input shape:", loaded_model.input_shape)
print("Loaded model output shape:", loaded_model.output_shape)
print("Loaded preprocessor type:", type(loaded_preprocessor))

Saved model loaded successfully.
Saved preprocessor loaded successfully.

Loaded model input shape: (None, 70)
Loaded model output shape: (None, 1)
Loaded preprocessor type: <class 'sklearn.compose._column_transformer.ColumnTransformer'>


In [22]:
# Step 20 — Verify a real prediction from the saved artifacts

# Verify prediction using the saved model + saved preprocessor

# Take one raw test sample
sample = X_test.iloc[[0]]

# Get the actual label
actual_label = y_test.iloc[0]

# Transform the raw sample using the SAVED preprocessor
sample_processed = loaded_preprocessor.transform(sample)

# Predict using the SAVED model
prediction_probability = loaded_model.predict(
    sample_processed,
    verbose=0
)[0][0]

prediction_label = int(prediction_probability >= 0.5)

print("Actual label:", actual_label)
print("Predicted probability:", prediction_probability)
print("Predicted label:", prediction_label)

Actual label: 0
Predicted probability: 3.4873627e-22
Predicted label: 0


In [23]:
# Step 21 — Create the prediction function

# Prediction function

def predict_loan_default(input_data):
    """
    Predict loan default status from the 32 original input features.

    Returns:
        0 = Loan Default
        1 = Not Loan Default
    """

    # Convert input into a DataFrame
    input_df = pd.DataFrame([input_data])

    # Preprocess using the saved preprocessor
    input_processed = loaded_preprocessor.transform(input_df)

    # Get prediction probability
    probability = loaded_model.predict(
        input_processed,
        verbose=0
    )[0][0]

    # Convert probability to binary class
    prediction = int(probability >= 0.5)

    return prediction


print("Prediction function created successfully.")

Prediction function created successfully.


In [24]:
# Step 22 — Test the prediction function with a real raw row

# Test the prediction function with a real dataset row

sample_input = X_test.iloc[0].to_dict()

prediction = predict_loan_default(sample_input)

print("Prediction:", prediction)

if prediction == 0:
    print("Result: Loan Default")
else:
    print("Result: Not Loan Default")

Prediction: 0
Result: Loan Default


In [67]:
print(X_test.iloc[:2,:])

          ID loan_limit Gender approv_in_adv loan_type loan_purpose  \
59366  84256         cf  Joint         nopre     type1           p3   
1052   25942         cf  Joint           pre     type1           p1   

      Credit_Worthiness open_credit business_or_commercial  loan_amount  ...  \
59366                l1        nopc                  nob/c       626500  ...   
1052                 l1        nopc                  nob/c       256500  ...   

        income  credit_type  Credit_Score  co-applicant_credit_type    age  \
59366  13080.0          CIB           596                       EXP  35-44   
1052    8220.0          CIB           846                       EXP  25-34   

      submission_of_application        LTV  Region Security_Type dtir1  
59366                  not_inst  47.897554   south        direct  25.0  
1052                    to_inst  89.062500   North        direct  46.0  

[2 rows x 32 columns]


In [25]:
# Step 23 — Create prediction.py

# Create prediction.py

prediction_code = '''
import os
import joblib
import pandas as pd
from tensorflow.keras.models import load_model


# Paths
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
MODEL_DIR = os.path.join(BASE_DIR, "loan_default_model")

MODEL_PATH = os.path.join(MODEL_DIR, "loan_default_nn.keras")
PREPROCESSOR_PATH = os.path.join(MODEL_DIR, "preprocessor.pkl")


# Load model and preprocessor once
model = load_model(MODEL_PATH)
preprocessor = joblib.load(PREPROCESSOR_PATH)


def predict_loan_default(input_data):
    """
    Predict loan default status.

    Input:
        Dictionary containing all 32 input features.

    Output:
        0 = Loan Default
        1 = Not Loan Default
    """

    # Convert input dictionary to DataFrame
    input_df = pd.DataFrame([input_data])

    # Apply the same preprocessing used during training
    input_processed = preprocessor.transform(input_df)

    # Predict probability
    probability = model.predict(
        input_processed,
        verbose=0
    )[0][0]

    # Convert probability to binary prediction
    prediction = int(probability >= 0.5)

    return prediction
'''

with open("prediction.py", "w") as f:
    f.write(prediction_code)

print("prediction.py created successfully.")
print("File exists:", os.path.exists("prediction.py"))

prediction.py created successfully.
File exists: True


In [26]:
# Step 24 — Test prediction.py independently

# Test the standalone prediction.py

import importlib.util

# Load prediction.py as a module
spec = importlib.util.spec_from_file_location(
    "prediction",
    "prediction.py"
)

prediction_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(prediction_module)

# Use the same real test sample
sample_input = X_test.iloc[0].to_dict()

# Call the function from prediction.py
prediction = prediction_module.predict_loan_default(sample_input)

print("Standalone prediction.py test successful.")
print("Prediction:", prediction)

if prediction == 0:
    print("Result: Loan Default")
else:
    print("Result: Not Loan Default")

Standalone prediction.py test successful.
Prediction: 0
Result: Loan Default


In [27]:
# Step 25 — Create the FastAPI application

# Create main.py for FastAPI

main_code = '''
from fastapi import FastAPI
from pydantic import BaseModel

from prediction import predict_loan_default


app = FastAPI(
    title="Loan Default Prediction API",
    description="Deep Learning API for Loan Default Prediction",
    version="1.0.0"
)


class LoanApplication(BaseModel):
    ID: int
    loan_limit: str
    Gender: str
    approv_in_adv: str
    loan_type: str
    loan_purpose: str
    Credit_Worthiness: str
    open_credit: str
    business_or_commercial: str
    loan_amount: float
    rate_of_interest: float | None = None
    Interest_rate_spread: float | None = None
    Upfront_charges: float | None = None
    term: float | None = None
    Neg_ammortization: str
    interest_only: str
    lump_sum_payment: str
    property_value: float | None = None
    construction_type: str
    occupancy_type: str
    Secured_by: str
    total_units: str
    income: float | None = None
    credit_type: str
    Credit_Score: int
    co_applicant_credit_type: str
    age: str
    submission_of_application: str
    LTV: float | None = None
    Region: str
    Security_Type: str
    dtir1: float | None = None


@app.get("/")
def home():
    return {
        "message": "Loan Default Prediction API is running"
    }


@app.post("/predict")
def predict(application: LoanApplication):

    input_data = application.model_dump()

    prediction = predict_loan_default(input_data)

    return {
        "prediction": prediction
    }
'''

with open("main.py", "w") as f:
    f.write(main_code)

print("main.py created successfully.")
print("File exists:", os.path.exists("main.py"))

main.py created successfully.
File exists: True


In [28]:
# Step 26 — Verify the FastAPI input fields

# Verify FastAPI input fields

import ast

with open("main.py", "r") as f:
    main_source = f.read()

tree = ast.parse(main_source)

loan_application_fields = []

for node in ast.walk(tree):
    if isinstance(node, ast.ClassDef) and node.name == "LoanApplication":
        for item in node.body:
            if isinstance(item, ast.AnnAssign):
                if isinstance(item.target, ast.Name):
                    loan_application_fields.append(item.target.id)

print("Number of API input fields:", len(loan_application_fields))

print("\nAPI input fields:")
for i, field in enumerate(loan_application_fields, start=1):
    print(f"{i}. {field}")

print("\nExpected number of input features:", X.shape[1])
print("Feature count matches:", len(loan_application_fields) == X.shape[1])

Number of API input fields: 32

API input fields:
1. ID
2. loan_limit
3. Gender
4. approv_in_adv
5. loan_type
6. loan_purpose
7. Credit_Worthiness
8. open_credit
9. business_or_commercial
10. loan_amount
11. rate_of_interest
12. Interest_rate_spread
13. Upfront_charges
14. term
15. Neg_ammortization
16. interest_only
17. lump_sum_payment
18. property_value
19. construction_type
20. occupancy_type
21. Secured_by
22. total_units
23. income
24. credit_type
25. Credit_Score
26. co_applicant_credit_type
27. age
28. submission_of_application
29. LTV
30. Region
31. Security_Type
32. dtir1

Expected number of input features: 32
Feature count matches: True


In [29]:
# Step 27 — Check FastAPI and Uvicorn

import fastapi
import uvicorn

print("FastAPI version:", fastapi.__version__)
print("Uvicorn version:", uvicorn.__version__)

FastAPI version: 0.136.1
Uvicorn version: 0.46.0


In [30]:
# Step 28 — Start the FastAPI server

import subprocess
import time

# Start FastAPI server
server_process = subprocess.Popen(
    [
        "uvicorn",
        "main:app",
        "--host", "0.0.0.0",
        "--port", "8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Give the server a few seconds to start
time.sleep(5)

print("FastAPI server started.")
print("Process ID:", server_process.pid)

FastAPI server started.
Process ID: 693


In [31]:
# Step 29 — Check the API health endpoint

import requests

response = requests.get("http://127.0.0.1:8000/")

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'message': 'Loan Default Prediction API is running'}


In [32]:
# Step 30 — Send a real prediction request to FastAPI

# Test the /predict endpoint with a real dataset row

# Take one raw test row
sample_input = X_test.iloc[0].to_dict()

# Convert NumPy values to normal Python values for JSON
sample_input = {
    key: (
        value.item() if isinstance(value, np.generic) else value
    )
    for key, value in sample_input.items()
}

# Send request to FastAPI
response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=sample_input
)

print("Status code:", response.status_code)
print("Response:", response.json())

print("\nActual label from dataset:", int(y_test.iloc[0]))

Status code: 422
Response: {'detail': [{'type': 'missing', 'loc': ['body', 'co_applicant_credit_type'], 'msg': 'Field required', 'input': {'ID': 84256, 'loan_limit': 'cf', 'Gender': 'Joint', 'approv_in_adv': 'nopre', 'loan_type': 'type1', 'loan_purpose': 'p3', 'Credit_Worthiness': 'l1', 'open_credit': 'nopc', 'business_or_commercial': 'nob/c', 'loan_amount': 626500, 'rate_of_interest': 3.75, 'Interest_rate_spread': -0.01, 'Upfront_charges': 661.5, 'term': 360.0, 'Neg_ammortization': 'not_neg', 'interest_only': 'not_int', 'lump_sum_payment': 'not_lpsm', 'property_value': 1308000.0, 'construction_type': 'sb', 'occupancy_type': 'pr', 'Secured_by': 'home', 'total_units': '1U', 'income': 13080.0, 'credit_type': 'CIB', 'Credit_Score': 596, 'co-applicant_credit_type': 'EXP', 'age': '35-44', 'submission_of_application': 'not_inst', 'LTV': 47.89755352, 'Region': 'south', 'Security_Type': 'direct', 'dtir1': 25.0}}]}

Actual label from dataset: 0


In [33]:
# Step 31 — Fix co-applicant_credit_type

# Fix the API field name to match the original dataset exactly

main_code = main_code.replace(
    "from pydantic import BaseModel",
    "from pydantic import BaseModel, Field"
)

main_code = main_code.replace(
    "    co_applicant_credit_type: str",
    '    co_applicant_credit_type: str = Field(alias="co-applicant_credit_type")'
)

# Allow Pydantic to work with both the Python field name and the dataset alias
main_code = main_code.replace(
    'app = FastAPI(',
    'app = FastAPI('
)

# Add model configuration inside LoanApplication
main_code = main_code.replace(
    'class LoanApplication(BaseModel):\n',
    '''class LoanApplication(BaseModel):
    model_config = {"populate_by_name": True}
'''
)

# Rewrite main.py
with open("main.py", "w") as f:
    f.write(main_code)

print("main.py updated successfully.")

main.py updated successfully.


In [34]:
# Step 32 — Restart FastAPI cleanly

# Stop the previous FastAPI server processes

import subprocess
import time

subprocess.run(
    ["pkill", "-f", "uvicorn"],
    capture_output=True,
    text=True
)

time.sleep(2)

# Start one clean FastAPI server
server_process = subprocess.Popen(
    [
        "uvicorn",
        "main:app",
        "--host", "0.0.0.0",
        "--port", "8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

print("FastAPI server restarted.")
print("Process ID:", server_process.pid)

FastAPI server restarted.
Process ID: 726


In [35]:
# Step 33 — Verify the restarted API

import requests

response = requests.get("http://127.0.0.1:8000/")

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'message': 'Loan Default Prediction API is running'}


In [37]:
# Step 34 — Test the actual /predict endpoint again

# Test the corrected /predict endpoint

sample_input = X_test.iloc[0].to_dict()

# Convert NumPy values to normal Python values
sample_input = {
    key: value.item() if isinstance(value, np.generic) else value
    for key, value in sample_input.items()
}

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=sample_input
)

print("Status code:", response.status_code)
print("Response:", response.json())

print("\nActual label from dataset:", int(y_test.iloc[0]))

Status code: 500


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [38]:
# Step 35 — Inspect the actual FastAPI error

# Inspect the raw FastAPI response

print("Status code:", response.status_code)
print("\nResponse headers:")
print(response.headers)

print("\nRaw response:")
print(response.text)

Status code: 500

Response headers:
{'date': 'Sat, 19 Sep 2026 16:40:07 GMT', 'server': 'uvicorn', 'content-length': '21', 'content-type': 'text/plain; charset=utf-8'}

Raw response:
Internal Server Error


In [39]:
# Step 36 — Read the Uvicorn error

# Read available FastAPI server output without waiting indefinitely

import time

time.sleep(1)

print("Server process running:", server_process.poll() is None)
print("Server process status:", server_process.poll())

Server process running: True
Server process status: None


In [40]:
# Step 36 — Start the server and immediately check whether it stays alive

import subprocess
import time
import os

# Start FastAPI server
log_file = open("fastapi.log", "w")

server_process = subprocess.Popen(
    [
        "uvicorn",
        "main:app",
        "--host", "127.0.0.1",
        "--port", "8000"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True
)

# Wait for startup
time.sleep(5)

print("Process ID:", server_process.pid)
print("Process running:", server_process.poll() is None)

# Show server log so far
log_file.flush()

with open("fastapi.log", "r") as f:
    print("\n--- FastAPI log ---")
    print(f.read())

Process ID: 758
Process running: True

--- FastAPI log ---



In [41]:
# Step 37 — Test /predict

# Test the /predict endpoint

sample_input = X_test.iloc[0].to_dict()

# Convert NumPy values to normal Python values
sample_input = {
    key: value.item() if isinstance(value, np.generic) else value
    for key, value in sample_input.items()
}

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=sample_input,
    timeout=30
)

print("Status code:", response.status_code)
print("Raw response:", response.text)

if response.headers.get("content-type", "").startswith("application/json"):
    print("JSON response:", response.json())

print("\nActual label from dataset:", int(y_test.iloc[0]))

Status code: 500
Raw response: Internal Server Error

Actual label from dataset: 0


In [42]:
# Step 38 — Restart FastAPI with unbuffered logs

# Stop the current server
server_process.terminate()
server_process.wait()

import subprocess
import time

# Start FastAPI with unbuffered output
server_process = subprocess.Popen(
    [
        "python", "-u", "-m", "uvicorn",
        "main:app",
        "--host", "127.0.0.1",
        "--port", "8000",
        "--log-level", "debug"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

time.sleep(3)

print("Process running:", server_process.poll() is None)

Process running: True


In [43]:
# Step 39 — Trigger /predict and capture the traceback

import requests
import time
import os

# Prepare one real test sample
sample_input = X_test.iloc[0].to_dict()

sample_input = {
    key: value.item() if isinstance(value, np.generic) else value
    for key, value in sample_input.items()
}

# Trigger the endpoint
response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=sample_input,
    timeout=30
)

print("HTTP status:", response.status_code)
print("Response:", response.text)

# Give Uvicorn a moment to write the traceback
time.sleep(1)

print("\n--- SERVER OUTPUT ---")

# Read whatever output is currently available
if server_process.stdout is not None:
    import select

    while True:
        ready, _, _ = select.select([server_process.stdout], [], [], 0)
        if not ready:
            break

        line = server_process.stdout.readline()

        if not line:
            break

        print(line, end="")

HTTP status: 500
Response: Internal Server Error

--- SERVER OUTPUT ---
2026-09-19 16:40:54.451848: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [44]:
# Step 40 — Fix main.py

# Fix the FastAPI prediction input to preserve the original dataset column names

main_code = main_code.replace(
    "input_data = application.model_dump()",
    "input_data = application.model_dump(by_alias=True)"
)

with open("main.py", "w") as f:
    f.write(main_code)

print("main.py updated successfully.")

# Verify the important line
for line in main_code.splitlines():
    if "input_data = application.model_dump" in line:
        print("Updated line:", line)

main.py updated successfully.
Updated line:     input_data = application.model_dump(by_alias=True)


In [45]:
# Step 41 — Restart FastAPI with the corrected main.py

# Stop the current FastAPI server
server_process.terminate()
server_process.wait()

import subprocess
import time

# Start FastAPI again with the corrected code
server_process = subprocess.Popen(
    [
        "python", "-u", "-m", "uvicorn",
        "main:app",
        "--host", "127.0.0.1",
        "--port", "8000",
        "--log-level", "debug"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

time.sleep(3)

print("FastAPI restarted.")
print("Process running:", server_process.poll() is None)

FastAPI restarted.
Process running: True


In [46]:
# Step 42 — Test /predict again

# Test the corrected /predict endpoint

sample_input = X_test.iloc[0].to_dict()

# Convert NumPy values to normal Python values
sample_input = {
    key: value.item() if isinstance(value, np.generic) else value
    for key, value in sample_input.items()
}

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=sample_input,
    timeout=30
)

print("Status code:", response.status_code)
print("Response:", response.text)

print("\nActual label from dataset:", int(y_test.iloc[0]))

Status code: 500
Response: Internal Server Error

Actual label from dataset: 0


In [68]:
# Save train/test datasets and the original dataset

df.to_csv("loan_default_dataset_processed.csv", index=False)
X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)

print("Files saved successfully:")
print("1. loan_default_dataset_processed.csv")
print("2. X_train.csv")
print("3. X_test.csv")

print("\nShapes:")
print("Full dataset:", df.shape)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

Files saved successfully:
1. loan_default_dataset_processed.csv
2. X_train.csv
3. X_test.csv

Shapes:
Full dataset: (148670, 33)
X_train: (118936, 32)
X_test: (29734, 32)


In [69]:
from IPython.display import FileLink

FileLink("loan_default_dataset_processed.csv")

/kaggle/working/loan_default_dataset_processed.csv

In [70]:
FileLink("X_test.csv")

/kaggle/working/X_test.csv

In [71]:
FileLink("X_train.csv")

/kaggle/working/X_train.csv

NameError: name 'y_test' is not defined